# Football Market Value — Notebook de démarrage complet

> **Problématique** : Quels facteurs influencent la valeur marchande d'un joueur de football ?

Ce notebook lance le projet de A à Z et vérifie que tout fonctionne.

## Ce que fait ce notebook
1. Démarre l'infrastructure Docker (8 services)
2. Upload les données CSV vers MinIO Bronze
3. Déclenche le pipeline Airflow et attend la fin
4. Vérifie la zone Silver (Parquet + partitionnement)
5. Vérifie la zone Gold (9 tables PostgreSQL)
6. Affiche les résultats métier
7. Vérifie la sécurité (rôles PostgreSQL)
8. Affiche le résumé complet

---

## Prérequis avant de lancer

- **Docker Desktop** installé et lancé
- **Données Kaggle** téléchargées dans `data/raw/`
- **Python** avec les dépendances installées : `pip install -r requirements.txt`

```
data/raw/
  Transfermarkt/
    players.csv
    player_valuations.csv
    appearances.csv
    clubs.csv
    competitions.csv
  FIFA 23 Players/
    male_players.csv
```

- [Transfermarkt sur Kaggle](https://www.kaggle.com/datasets/davidcariboo/player-scores)
- [FIFA 23 Players sur Kaggle](https://www.kaggle.com/datasets/stefanoleone992/fifa-23-complete-player-dataset)

---
## Étape 0 — Chargement des librairies

On importe toutes les librairies nécessaires :
- `subprocess` : pour lancer des commandes Docker depuis Python
- `boto3` : pour parler à MinIO (compatible API Amazon S3)
- `psycopg2` : pour se connecter à PostgreSQL
- `pandas` : pour afficher les résultats proprement
- `requests` : pour appeler l'API REST d'Airflow

In [ ]:
import subprocess
import time
import boto3
import psycopg2
import pandas as pd
import requests
from pathlib import Path
from botocore.client import Config
import warnings
warnings.filterwarnings('ignore')

# Configuration globale
AIRFLOW_URL  = 'http://localhost:8081'
MINIO_URL    = 'http://localhost:9002'
DAG_ID       = 'football_market_value_pipeline'
AUTH         = ('admin', 'admin')

print('Librairies chargees avec succes !')
print(f'Airflow  : {AIRFLOW_URL}')
print(f'MinIO    : {MINIO_URL}')
print(f'Spark UI : http://localhost:8080')
print(f'Metabase : http://localhost:3000')

---
## Étape 1 — Démarrage de l'infrastructure Docker

### Architecture des 8 services

```
football_net (réseau Docker privé)
  ├── postgres        : PostgreSQL 15 (zone Gold + meta Airflow)
  ├── minio           : MinIO S3 (zone Bronze + Silver)
  ├── spark-master    : Spark Master (coordination)
  ├── spark-worker    : Spark Worker (2 cores, 2 GB RAM)
  ├── airflow-init    : Initialisation Airflow (une seule fois)
  ├── airflow-webserver : Interface web Airflow (port 8081)
  ├── airflow-scheduler : Planificateur DAG
  └── metabase        : Dashboards BI (port 3000)
```

**Pourquoi Docker ?** Reproductibilité totale — même environnement sur n'importe quelle machine. Une seule commande lance tout.

In [ ]:
print('Lancement de docker-compose...')
result = subprocess.run(
    ['docker-compose', 'up', '-d'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('docker-compose up -d : OK')
else:
    print('ERREUR :', result.stderr)
    raise Exception('docker-compose a echoue. Verifier que Docker Desktop est lance.')

In [ ]:
print('Etat des conteneurs Docker :')
result = subprocess.run(['docker-compose', 'ps'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
print('Attente du demarrage complet des services...')
print('(PostgreSQL, MinIO, Airflow peuvent prendre 2-3 minutes)')
print()

# Attente minimale
time.sleep(30)
print('30s ecoules — verification Airflow en cours...')

# Attente dynamique Airflow (max 4 minutes)
for attempt in range(24):
    try:
        resp = requests.get(f'{AIRFLOW_URL}/health', timeout=5)
        if resp.status_code == 200:
            scheduler_status = resp.json().get('scheduler', {}).get('status', '?')
            print(f'Airflow pret ! Scheduler : {scheduler_status}')
            break
    except Exception:
        pass
    print(f'  [{(attempt+1)*10 + 30}s] Airflow pas encore pret...')
    time.sleep(10)
else:
    print('ATTENTION : Airflow pas reponsif apres 4min')
    print('Verifier : docker-compose logs airflow-webserver')

---
## Étape 2 — Upload des données vers Bronze (MinIO)

### Pourquoi MinIO ?

MinIO est compatible avec l'API Amazon S3 — le standard du cloud. Le même code Spark qui lit depuis `s3a://bronze/...` fonctionnerait sur AWS S3 en production sans modification.

### Pourquoi l'upload est séparé du DAG ?

Les données Kaggle sont statiques — elles ne changent pas tous les jours. Intégrer l'upload dans le DAG signifierait re-uploader 6.8 GB à chaque run quotidien, ce qui est inutile. En production avec des données dynamiques (API temps réel), cet upload serait une tâche Airflow.

### Zone Bronze — Règle fondamentale

Bronze est **intouchable**. On n'y écrit qu'une seule fois, on ne modifie jamais. Si Spark échoue, on repart de Bronze sans perdre les données brutes.

In [ ]:
print('Connexion a MinIO (S3-compatible)...')

# Connexion boto3 vers MinIO
# boto3 = librairie officielle Amazon pour parler a S3 et compatibles
# signature_version=s3v4 = version du protocole obligatoire pour MinIO
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_URL,
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Attente que MinIO soit pret (max 60s)
for attempt in range(6):
    try:
        s3.list_buckets()
        print('MinIO accessible !')
        break
    except Exception as e:
        print(f'  MinIO pas encore pret ({attempt+1}/6), attente 10s...')
        time.sleep(10)
else:
    raise Exception('MinIO inaccessible apres 60s. Verifier docker-compose ps')

# Creer les buckets si manquants
# Bronze = données brutes CSV / Silver = données nettoyées Parquet
for bucket in ['bronze', 'silver']:
    existing = [b['Name'] for b in s3.list_buckets()['Buckets']]
    if bucket not in existing:
        s3.create_bucket(Bucket=bucket)
        print(f'Bucket cree : {bucket}')
    else:
        print(f'Bucket existant : {bucket}')

In [ ]:
print('Upload des fichiers CSV vers Bronze...')
print()

raw_path = Path('data/raw')
csv_files = list(raw_path.rglob('*.csv'))

if not csv_files:
    raise Exception(
        'Aucun fichier CSV trouve dans data/raw/.\n'
        'Telecharger les datasets Kaggle et les placer dans data/raw/'
    )

print(f'{len(csv_files)} fichiers CSV trouves')
print()

uploaded = 0
already_exists = 0

for file_path in csv_files:
    relative = file_path.relative_to(raw_path)
    # Correction backslash Windows -> slash S3
    s3_key = str(relative).replace('\\', '/')

    try:
        s3.head_object(Bucket='bronze', Key=s3_key)
        print(f'  EXISTE DEJA : {s3_key}')
        already_exists += 1
    except Exception:
        s3.upload_file(str(file_path), 'bronze', s3_key)
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f'  UPLOADE : {s3_key} ({size_mb:.1f} MB)')
        uploaded += 1

print()
print(f'Upload termine : {uploaded} nouveaux fichiers, {already_exists} deja presents')

In [ ]:
print('Verification des fichiers dans Bronze :')
print()

response = s3.list_objects_v2(Bucket='bronze')
fichiers = response.get('Contents', [])

if not fichiers:
    raise Exception('Bronze est vide ! Verifier upload_to_bronze.py')

print(f'Nombre de fichiers dans Bronze : {len(fichiers)}')
print()
total_mb = 0
for f in fichiers:
    taille_mb = f['Size'] / (1024 * 1024)
    total_mb += taille_mb
    print(f"  {f['Key']:<60} {taille_mb:>8.1f} MB")

print()
print(f'Taille totale Bronze : {total_mb:.0f} MB')

---
## Étape 3 — Déclenchement du pipeline Airflow

### Architecture du DAG

```
check_minio_buckets
       |
       v
check_data_available   <- verifie CSV presents et non vides
       |
       v
check_postgres         <- verifie PostgreSQL accessible avant Spark
       |
       v
bronze_to_silver       <- Spark Job 1 : CSV -> Parquet + nettoyage
       |
       v
silver_to_gold         <- Spark Job 2 : Parquet -> PostgreSQL
       |
       v
pipeline_success       <- verifie counts finaux + confirmation
```

**Pourquoi Airflow et pas un simple cron ?** Airflow gère les dépendances entre tâches (silver_to_gold ne démarre pas si bronze_to_silver a échoué), les retries automatiques, et fournit un monitoring complet via son interface web.

**Suivi en temps réel** : http://localhost:8081

In [ ]:
print('Declenchement du DAG Airflow...')

# Activer le DAG (au cas ou il soit en pause)
resp_patch = requests.patch(
    f'{AIRFLOW_URL}/api/v1/dags/{DAG_ID}',
    json={'is_paused': False},
    auth=AUTH
)
print(f'DAG active : HTTP {resp_patch.status_code}')

# Declencher le run
resp_run = requests.post(
    f'{AIRFLOW_URL}/api/v1/dags/{DAG_ID}/dagRuns',
    json={},
    auth=AUTH
)
print(f'DAG declenche : HTTP {resp_run.status_code}')

if resp_run.status_code == 200:
    run_id = resp_run.json().get('dag_run_id')
    print(f'Run ID : {run_id}')
    print()
    print(f'Suivi en direct : {AIRFLOW_URL}')
    print('Toutes les taches doivent devenir vertes.')
else:
    print('ERREUR :', resp_run.text)
    raise Exception('Impossible de declencher le DAG')

In [ ]:
print('Attente de la fin du pipeline (10-20 minutes)...')
print()
print('Etapes en cours :')
print('  1. check_minio_buckets   -> verifie MinIO')
print('  2. check_data_available  -> verifie les CSV')
print('  3. check_postgres        -> verifie PostgreSQL')
print('  4. bronze_to_silver      -> Spark nettoie et convertit en Parquet')
print('  5. silver_to_gold        -> Spark joint et ecrit dans PostgreSQL')
print('  6. pipeline_success      -> verification finale')
print()

max_wait = 80  # 80 x 30s = 40 minutes max
for i in range(max_wait):
    time.sleep(30)

    response = requests.get(
        f'{AIRFLOW_URL}/api/v1/dags/{DAG_ID}/dagRuns?limit=1&order_by=-start_date',
        auth=AUTH
    )

    if response.status_code == 200:
        runs = response.json().get('dag_runs', [])
        if runs:
            state = runs[0].get('state')
            elapsed = (i + 1) * 30
            print(f'  [{elapsed}s] Status : {state}')

            if state == 'success':
                print()
                print('PIPELINE TERMINE AVEC SUCCES !')
                break
            elif state == 'failed':
                print()
                print('PIPELINE ECHOUE')
                print(f'Voir les logs : {AIRFLOW_URL}')
                raise Exception('Le pipeline Airflow a echoue. Consulter les logs Airflow.')
else:
    print('Timeout (40min). Verifier manuellement dans Airflow.')

---
## Étape 4 — Vérification Zone Silver (MinIO Parquet)

### Pourquoi Parquet ?

Parquet est un format **columnar** — il stocke les données par colonne et non par ligne.
- Si Spark a besoin de l'âge moyen des joueurs, il lit UNIQUEMENT la colonne `age`
- CSV = Spark lirait toute la ligne (player_id, name, age, position, value...)
- Résultat : **5x moins de stockage**, **10x plus rapide**

### Pourquoi le partitionnement par année ?

On a créé une colonne `valuation_year` depuis la date, puis partitionné sur cette colonne. Spark crée automatiquement des sous-dossiers :
```
silver/player_valuations/valuation_year=2019/
silver/player_valuations/valuation_year=2020/
...
```
Si on filtre sur 2023, Spark lit **uniquement** `valuation_year=2023/` — c'est le **Predicate Pushdown**, 8x plus rapide.

In [ ]:
print('VERIFICATION ZONE SILVER (MinIO Parquet)')
print('=' * 55)
print()

# Lister les tables Parquet dans Silver
response = s3.list_objects_v2(Bucket='silver', Delimiter='/')
prefixes = response.get('CommonPrefixes', [])

if not prefixes:
    print('ATTENTION : Silver est vide ! Le Job 1 Spark a peut-etre echoue.')
else:
    print('Tables Parquet dans Silver :')
    for p in prefixes:
        prefix = p['Prefix']
        obj_resp = s3.list_objects_v2(Bucket='silver', Prefix=prefix)
        nb_fichiers = len(obj_resp.get('Contents', []))
        # Calcul taille
        taille_mb = sum(o['Size'] for o in obj_resp.get('Contents', [])) / (1024*1024)
        print(f'  {prefix:<35} {nb_fichiers} fichiers  {taille_mb:.1f} MB')

print()
print('Partitionnement par annee (valuation_year) :')
response_part = s3.list_objects_v2(
    Bucket='silver',
    Prefix='player_valuations/',
    Delimiter='/'
)
partitions = response_part.get('CommonPrefixes', [])
for p in partitions:
    print(f'  {p["Prefix"]}')

print()
print(f'Nombre de partitions annees : {len(partitions)}')
print('Predicate Pushdown actif : filtre annee -> 1 dossier lu au lieu de tout')

---
## Étape 5 — Vérification Zone Gold (PostgreSQL)

### Modèle de données — Schéma en étoile

```
         dim_player (47 637)     <- QUI est le joueur ?
              ^
              | player_id
              |
dim_club <----+----> fact_player_value <----> dim_competition
(796)         |      (31 507 lignes)          (67)
              |      COMBIEN vaut-il ?
              v
    5 tables agregees (5 a 50 lignes)
    pre-calculees pour Metabase
```

**Pourquoi le schéma en étoile ?** Optimisé pour les requêtes analytiques — peu de jointures, lecture rapide, structure simple. Metabase peut interroger directement `agg_value_by_position` (5 lignes) au lieu de recalculer sur `fact_player_value` (31 507 lignes).

In [ ]:
print('Connexion a PostgreSQL (zone Gold)...')

try:
    conn = psycopg2.connect(
        host='localhost', port=5432,
        database='football',
        user='airflow', password='airflow',
        connect_timeout=10
    )
    print('Connecte a PostgreSQL/football !')
except Exception as e:
    raise Exception(f'Impossible de se connecter a PostgreSQL : {e}')

cursor = conn.cursor()

tables = [
    ('dim_player',              'Dimension joueurs (enrichi FIFA 23)'),
    ('dim_club',                'Dimension clubs'),
    ('dim_competition',         'Dimension championnats'),
    ('fact_player_value',       'TABLE DE FAITS CENTRALE'),
    ('agg_value_by_position',   'Agregation par poste'),
    ('agg_value_by_league',     'Agregation par championnat'),
    ('agg_value_by_age',        'Agregation par age'),
    ('agg_top_players',         'Top 50 joueurs'),
    ('agg_value_by_nationality','Agregation par nationalite'),
]

print()
print(f'{"Table":<35} {"Lignes":>8}   Description')
print('-' * 80)

total = 0
for table, description in tables:
    try:
        cursor.execute(f'SELECT COUNT(*) FROM {table}')
        count = cursor.fetchone()[0]
        total += count
        marker = '  <-- TABLE CENTRALE' if table == 'fact_player_value' else ''
        print(f'{table:<35} {count:>8,}   {description}{marker}')
    except Exception as e:
        print(f'{table:<35} ERREUR : {e}')

print('-' * 80)
print(f'{"TOTAL":<35} {total:>8,}')

---
## Étape 6 — Résultats métier

Ces requêtes répondent directement à la problématique : **quels facteurs influencent la valeur marchande ?**

On interroge les tables pré-agrégées — résultats instantanés.

**Note sur les valeurs** : les moyennes semblent basses car notre dataset contient tous les niveaux de jeu — de la Ligue 1 française aux divisions inférieures européennes. Les tops joueurs ont des valeurs > 100M€.

In [ ]:
print('FACTEUR 1 : LE POSTE')
print('Question : Les attaquants valent-ils plus que les defenseurs ?')
print('=' * 60)
print()

# On lit depuis agg_value_by_position (5 lignes) et non fact_player_value (31507)
# C'est la pre-agregation en action
df_pos = pd.read_sql("""
    SELECT
        position,
        player_count,
        ROUND(avg_market_value) AS valeur_moyenne_eur,
        ROUND(max_market_value) AS valeur_max_eur,
        ROUND(avg_fifa_rating, 1) AS note_fifa_moyenne
    FROM agg_value_by_position
    WHERE position IS NOT NULL
    ORDER BY avg_market_value DESC
""", conn)

df_pos['valeur_moyenne_eur'] = df_pos['valeur_moyenne_eur'].apply(lambda x: f"{x:,.0f} EUR")
df_pos['valeur_max_eur'] = df_pos['valeur_max_eur'].apply(lambda x: f"{x:,.0f} EUR")
print(df_pos.to_string(index=False))
print()
print('REPONSE : OUI — les attaquants sont les plus valorises')
print('Source : agg_value_by_position (5 lignes) au lieu de fact_player_value (31 507)')

In [ ]:
print('FACTEUR 2 : L\'AGE')
print('Question : A quel age un joueur atteint-il son pic de valeur ?')
print('=' * 60)
print()

df_age = pd.read_sql("""
    SELECT
        age,
        player_count,
        ROUND(avg_market_value) AS valeur_moyenne_eur
    FROM agg_value_by_age
    ORDER BY avg_market_value DESC
    LIMIT 8
""", conn)

df_age['valeur_moyenne_eur'] = df_age['valeur_moyenne_eur'].apply(lambda x: f"{x:,.0f} EUR")
print('Top 8 ages par valeur marchande moyenne :')
print(df_age.to_string(index=False))
print()
print('REPONSE : Pic de valeur entre 24 et 26 ans')
print('Avant : joueur en developpement, valeur croissante')
print('Apres 28 ans : les clubs anticipent la fin de carriere')
print()
print('Note : le pic apparent a 18-19 ans est un biais de selection —')
print('seuls les talents exceptionnels ont une valorisation a cet age')

In [ ]:
print('FACTEUR 3 : LE CHAMPIONNAT')
print('Question : Quel championnat a les joueurs les plus valorises ?')
print('=' * 60)
print()

df_league = pd.read_sql("""
    SELECT
        competition_name,
        league_country,
        player_count,
        ROUND(avg_market_value) AS valeur_moyenne_eur,
        ROUND(avg_fifa_rating, 1) AS note_fifa_moyenne
    FROM agg_value_by_league
    ORDER BY avg_market_value DESC
    LIMIT 8
""", conn)

df_league['valeur_moyenne_eur'] = df_league['valeur_moyenne_eur'].apply(lambda x: f"{x:,.0f} EUR")
print(df_league.to_string(index=False))
print()
print('REPONSE : La Premier League (Angleterre) domine')
print('Les 5 grands championnats europeens concentrent les joueurs les plus chers')

In [ ]:
print('FACTEUR 4 : LA NOTE FIFA')
print('Question : La note FIFA reflete-t-elle la valeur marchande reelle ?')
print('=' * 65)
print()

# Correlation entre note FIFA et valeur Transfermarkt
# On requete directement sur fact_player_value pour ce calcul
df_fifa = pd.read_sql("""
    SELECT
        CASE
            WHEN overall_rating >= 85 THEN '1. Elite (85+)'
            WHEN overall_rating >= 75 THEN '2. Bon (75-84)'
            WHEN overall_rating >= 65 THEN '3. Moyen (65-74)'
            ELSE '4. Faible (moins de 65)'
        END AS niveau_fifa,
        COUNT(*) AS nb_joueurs,
        ROUND(AVG(latest_market_value)) AS valeur_moyenne_eur
    FROM fact_player_value
    WHERE overall_rating IS NOT NULL
    GROUP BY niveau_fifa
    ORDER BY niveau_fifa
""", conn)

df_fifa['valeur_moyenne_eur'] = df_fifa['valeur_moyenne_eur'].apply(lambda x: f"{x:,.0f} EUR")
print(df_fifa.to_string(index=False))
print()
print('REPONSE : OUI — correlation confirmee entre note FIFA et valeur reelle')
print('Un joueur Elite (85+) vaut en moyenne bien plus qu\'un joueur Faible (<65)')
print('Croisement des 2 sources Transfermarkt + FIFA 23 valide notre approche')

In [ ]:
print('TOP 10 JOUEURS PAR VALEUR MARCHANDE')
print('=' * 75)
print()

df_top = pd.read_sql("""
    SELECT
        player_name,
        position,
        nationality,
        age,
        latest_market_value,
        competition_name,
        overall_rating
    FROM fact_player_value
    ORDER BY latest_market_value DESC
    LIMIT 10
""", conn)

df_top['latest_market_value'] = df_top['latest_market_value'].apply(lambda x: f"{x:,.0f} EUR")
print(df_top.to_string(index=False))

---
## Étape 7 — Vérification Sécurité

### Principe du moindre privilège

Deux rôles PostgreSQL créés automatiquement dans `init_db.sql` :

| Rôle | Droits | Utilisé par |
|------|--------|-------------|
| `role_spark_etl` | SELECT + INSERT + UPDATE + DELETE | Apache Spark |
| `role_metabase_read` | SELECT uniquement | Metabase |

Si Metabase est compromis, un attaquant peut **lire** mais ne peut **pas modifier** les données.

On va tester ça en live :

In [ ]:
print('VERIFICATION DES ROLES POSTGRESQL')
print('=' * 55)
print()

# Verification que les roles existent
cursor.execute("""
    SELECT rolname, rolcanlogin
    FROM pg_roles
    WHERE rolname IN ('role_spark_etl', 'role_metabase_read')
    ORDER BY rolname
""")
roles = cursor.fetchall()

print('Roles existants :')
for role in roles:
    print(f'  {role[0]:<25} peut_se_connecter={role[1]}')

if len(roles) < 2:
    print()
    print('ATTENTION : roles manquants ! Verifier init_db.sql')
else:
    print()
    print('Les 2 roles sont presents OK')

In [ ]:
print('TEST role_metabase_read (lecture seule) :')
print()

try:
    conn_read = psycopg2.connect(
        host='localhost', port=5432,
        database='football',
        user='role_metabase_read',
        password='metabase_read_2024',
        connect_timeout=10
    )
    cursor_read = conn_read.cursor()

    # Test 1 : SELECT doit fonctionner
    cursor_read.execute('SELECT COUNT(*) FROM fact_player_value')
    count = cursor_read.fetchone()[0]
    print(f'  SELECT OK : {count:,} lignes lues depuis fact_player_value')

    # Test 2 : DELETE doit echouer
    print('  Test DELETE (doit etre refuse) :')
    try:
        cursor_read.execute('DELETE FROM fact_player_value WHERE player_id = -999')
        conn_read.rollback()
        print('  DELETE : AUTORISE (probleme de securite !)')
    except psycopg2.errors.InsufficientPrivilege:
        conn_read.rollback()
        print('  DELETE REFUSE : securite OK !')

    # Test 3 : INSERT doit echouer
    print('  Test INSERT (doit etre refuse) :')
    try:
        cursor_read.execute("INSERT INTO agg_value_by_position (position) VALUES ('test')")
        conn_read.rollback()
        print('  INSERT : AUTORISE (probleme de securite !)')
    except psycopg2.errors.InsufficientPrivilege:
        conn_read.rollback()
        print('  INSERT REFUSE : securite OK !')

    conn_read.close()

except Exception as e:
    print(f'  Impossible de se connecter avec role_metabase_read : {e}')

---
## Étape 8 — Résumé final

Récapitulatif complet de l'état du pipeline.

In [ ]:
print('=' * 65)
print('RESUME COMPLET — Football Market Value Pipeline')
print('=' * 65)
print()

print('INFRASTRUCTURE DOCKER')
result = subprocess.run(['docker-compose', 'ps', '--format', 'table'], capture_output=True, text=True)
# Compter les services running
lines = [l for l in result.stdout.split('\n') if 'running' in l.lower() or 'Up' in l]
print(f'  Services actifs : {len(lines)}/8')
print()

print('ZONE BRONZE (MinIO CSV)')
resp = s3.list_objects_v2(Bucket='bronze')
nb_bronze = len(resp.get('Contents', []))
taille_bronze = sum(o['Size'] for o in resp.get('Contents', [])) / (1024*1024)
print(f'  Fichiers : {nb_bronze}')
print(f'  Taille   : {taille_bronze:.0f} MB')
print()

print('ZONE SILVER (MinIO Parquet)')
resp_s = s3.list_objects_v2(Bucket='silver', Delimiter='/')
nb_tables = len(resp_s.get('CommonPrefixes', []))
resp_part = s3.list_objects_v2(Bucket='silver', Prefix='player_valuations/', Delimiter='/')
nb_partitions = len(resp_part.get('CommonPrefixes', []))
print(f'  Tables Parquet     : {nb_tables}')
print(f'  Partitions annees  : {nb_partitions} (valuation_year=2016 a 2023)')
print()

print('ZONE GOLD (PostgreSQL)')
total_lignes = 0
tables_list = [
    'dim_player', 'dim_club', 'dim_competition', 'fact_player_value',
    'agg_value_by_position', 'agg_value_by_league', 'agg_value_by_age',
    'agg_top_players', 'agg_value_by_nationality'
]
for t in tables_list:
    try:
        cursor.execute(f'SELECT COUNT(*) FROM {t}')
        c = cursor.fetchone()[0]
        total_lignes += c
        print(f'  {t:<35} {c:>8,} lignes')
    except:
        print(f'  {t:<35} MANQUANTE')
print(f'  {"TOTAL":<35} {total_lignes:>8,} lignes')
print()

print('SECURITE')
cursor.execute("SELECT COUNT(*) FROM pg_roles WHERE rolname IN ('role_spark_etl', 'role_metabase_read')")
nb_roles = cursor.fetchone()[0]
print(f'  Roles PostgreSQL : {nb_roles}/2 (moindre privilege)')
print()

print('REPONSES A LA PROBLEMATIQUE')
print('  Facteur 1 - Poste       : Attack > Midfield > Defender > Goalkeeper')
print('  Facteur 2 - Age         : Pic entre 24 et 26 ans')
print('  Facteur 3 - Championnat : Premier League domine')
print('  Facteur 4 - Note FIFA   : Correlation confirmee avec valeur reelle')
print()

print('INTERFACES DISPONIBLES')
print(f'  Airflow  : {AIRFLOW_URL}   (admin / admin)')
print('  MinIO    : http://localhost:9003  (minioadmin / minioadmin)')
print('  Spark UI : http://localhost:8080')
print('  Metabase : http://localhost:3000')
print('  PostgreSQL : localhost:5432 (airflow / airflow)')
print()

print('=' * 65)
print('PIPELINE COMPLET ET FONCTIONNEL')
print('=' * 65)

conn.close()